In [ ]:
import pandas as pd
import json
from IPython.display import display, Markdown, Latex
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, MOB_LOCATION_SOURCE_MAP, MOB_LOCATION_NAME_MAP, CONT_CMAP, FULL_RUN, TIMEOUTS, RERUNS
from emu_renewal.plotting import compare_proc_mob, compare_proc_weighted_gmob
from emu_renewal.utils import get_countries_by_continent, split_list_into_segments, get_analysis_paths, get_analysis_commits_df

set_matplotlib_formats("svg")

In [ ]:
all_countries = json.load(open(DATA_PATH / "config/included.json", "r"))
analysis_paths = get_analysis_paths(RERUNS + FULL_RUN + TIMEOUTS, all_countries)
countries_by_cont = get_countries_by_continent(all_countries)
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

# Purpose
This document presents the residual transmission scaling (with uncertainty) implemented without scaling for mobility for each country. It is intended to provide a raw comparison between the residual non-mechanistic transmission that needed to be applied for each analysis in the absence of mobility scaling and the various mobility data fields available from both Google and Facebook. We also show comparisons against the composite Google mobility metric after weighting, with associated credible intervals.

In [ ]:
display(Markdown("\\newpage # Individual mobility metric comparisons (Google and Facebook)\n\n"))
for m, mob_type in enumerate(MOB_LOCATION_SOURCE_MAP):
    mob_source = MOB_LOCATION_SOURCE_MAP[mob_type]
    mob_name = MOB_LOCATION_NAME_MAP[mob_type]
    if m:
        display(Latex(r"\newpage"))
    section_title = f"## {mob_name} mobility metric comparison\n\n"
    display(Markdown(section_title))
    display(Markdown(notes[mob_source]))
    mob_avail_countries = [c for c in all_countries if mob_source in analysis_paths[c]]
    for cont in CONT_CMAP:
        cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
        display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
        for countries in split_list_into_segments(cont_countries, 12):
            display(compare_proc_mob(analysis_paths, countries, 4, mob_type))

mob_avail_countries = [c for c in all_countries if "g_mob" in analysis_paths[c]]
display(Markdown("# Weighted scaling process versus residual comparison\n\n"))
for cont in CONT_CMAP:
    display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
    
    cont_countries = [c for c in countries_by_cont[cont] if c in mob_avail_countries]
    for countries in split_list_into_segments(cont_countries, 16):
        display(compare_proc_weighted_gmob(analysis_paths, countries, 200, 4))

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())